# Running in Colab

Open this notebook from GitHub and run **Runtime menu, then Run all**. The first cell prepares the current Colab runtime, mounts Drive, installs the project, checks out the pinned MIGRES code, and loads `GEMINI_API_KEY` from Colab Secrets. Keep using this same notebook/runtime through parameter selection. Data and outputs are stored in `MyDrive/SAR-MIGRES`; the notebook code is read from GitHub.
# 01 — Parameter and Hyperparameter Selection

Selects the State Aware Retrieval weights **α, β, γ** exactly as specified in the project report (§5.6.1, Tables 5.2–5.3).

| Item | Value (from the report) |
|---|---|
| Datasets | **HotpotQA, MuSiQue, Natural Questions** (§5.5.1) |
| Tuned | α, β, γ only — non-negative, sum to 1 (Eq. 8), so two free values |
| Fixed, **not** tuned | **K = 50** (candidate pool of the original MIGRES retriever); max iterations = 5 |
| Search | Gaussian-process Bayesian optimization |
| Trials | **10 in total = 3 random + 7 optimizer-proposed** |
| One trial | SAR inside the full MIGRES loop on **8 validation questions from each dataset** (24 questions); score = mean Recall@K |
| Seed | one fixed seed (13) for the search and for the 10% holdout |
| Result | the weights of the best trial are selected, fixed, and used in all later experiments |

Validation data only: questions come from a deterministic 10% holdout of each *training* split; public dev is untouched.

**Run order:** conformance check → section 0 (packages, models, datasets, corpus, labeled tuning questions) → tuning. The result is written to `outputs/experiments/01_parameter_selection/selected_config.json`, which notebooks 02, 03, 06 and 07 read.

In [ ]:
# Colab startup: initialize this notebook's runtime and persistent Drive folders
from pathlib import Path
import os, subprocess, sys

PROJECT_REPO_URL = "https://github.com/KunalDahal/State-Aware-Retrieval.git"
PROJECT_BRANCH = "master"
PROJECT_DIR = Path("/content/SAR")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/SAR-MIGRES")
MIGRES_DIR = PROJECT_DIR / "src" / "MIGRES"
MIGRES_URL = "https://github.com/AdelWang/MIGRES.git"
EXPECTED_MIGRES_COMMIT = "1a67b8769ca3fdf2b497e109e529143822248ec5"

# Clone the project if this runtime does not already contain a valid checkout.
if not (PROJECT_DIR / "pyproject.toml").exists():
    if PROJECT_DIR.exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a valid checkout. Remove it in Colab's Files pane, then rerun this cell.")
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", PROJECT_BRANCH, PROJECT_REPO_URL, str(PROJECT_DIR)], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f"Project clone failed:\n{clone.stderr or clone.stdout}")
os.chdir(PROJECT_DIR)

# Keep large/generated state in Drive across Colab runtime resets.
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
for name in ("data", "outputs"):
    src, dst = PROJECT_DIR / name, DRIVE_PROJECT_DIR / name
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink() and src.resolve() == dst.resolve():
        continue
    if src.exists() and not src.is_symlink():
        for item in src.iterdir():
            target = dst / item.name
            if not target.exists():
                item.rename(target)
        src.rmdir()
    if not src.exists():
        src.symlink_to(dst, target_is_directory=True)

# Install the same project extras required by the experiment notebooks.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[all]"], check=True)

# Colab may retain a torchvision binary built for a different torch version.
# This experiment uses text encoders only; install the matching current torchvision wheel.
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "torchvision"], check=True)

# The adapter expects this exact upstream MIGRES revision.
if not (MIGRES_DIR / ".git").exists():
    if MIGRES_DIR.exists() and any(MIGRES_DIR.iterdir()):
        raise RuntimeError(f"{MIGRES_DIR} exists but is not a Git checkout. Remove it from Colab's Files pane and rerun.")
    subprocess.run(["git", "clone", MIGRES_URL, str(MIGRES_DIR)], check=True)
subprocess.run(["git", "-C", str(MIGRES_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
subprocess.run(["git", "-C", str(MIGRES_DIR), "checkout", "--detach", EXPECTED_MIGRES_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(MIGRES_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != EXPECTED_MIGRES_COMMIT:
    raise RuntimeError(f"MIGRES commit mismatch: expected {EXPECTED_MIGRES_COMMIT}, got {actual_commit}")
os.environ["MIGRES_REPO_DIR"] = str(MIGRES_DIR)

# Colab Secrets: add GEMINI_API_KEY in the key icon in Colab's left sidebar.
try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets, enable notebook access, then rerun this cell.") from exc
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is empty in Colab Secrets. Add it and enable notebook access.")
os.environ["GEMINI_API_KEY"] = api_key
os.environ.setdefault("OPENAI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")
print("Project:", PROJECT_DIR)
print("Persistent data/output:", DRIVE_PROJECT_DIR)
print("Pinned MIGRES:", actual_commit)
print("Gemini key: loaded from Colab Secrets")


In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "src" / "configs" / "experiments.yaml").read_text())
# default.yaml + gemini.yaml: the Gemini overlay selects gemini-3.5-flash-lite through the OpenAI-compatible
# endpoint with the provider-default temperature (report Sec. 5.5.2 / 7.2.3). default.yaml alone does NOT do this.
BASE_CONFIG = [ROOT / "src" / "configs" / "default.yaml", ROOT / "src" / "configs" / "gemini.yaml"]

## Report-conformance check

Every value that the report fixes is declared once in `SPEC` and checked against `src/configs/` before anything expensive runs. If a config drifts from the report, this cell stops the notebook.

In [ ]:
# Make the src-layout package importable in a fresh Colab runtime.
import sys
from pathlib import Path

if "ROOT" not in globals() or not (Path(ROOT) / "pyproject.toml").exists():
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Project checkout not found. Run the Colab startup cell first.")
ROOT = Path(ROOT)
SRC_DIR = ROOT / "src"
if not (SRC_DIR / "sar_migres").is_dir():
    raise RuntimeError(f"sar_migres package folder not found at {SRC_DIR / 'sar_migres'}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from sar_migres.utils.config import load_config

SPEC = dict(                       # values fixed by the report
    datasets=["hotpotqa", "musique", "natural_questions"],   # Sec. 5.5.1
    k=50, t_max=5,                                            # Table 5.3 (same as original MIGRES)
    n_init=3, n_iter=7,                                       # Sec. 5.6.1: 3 random + 7 optimizer trials
    n_per_dataset=8,                                          # Sec. 5.6.1
    llm_model="gemini-3.5-flash-lite",                        # Table 5.1
    embedding_model="sentence-transformers/multi-qa-mpnet-base-cos-v1",
)
_cfg = load_config(*BASE_CONFIG)
_ps = PLAN["parameter_selection"]; _bo = _ps["bayesian_optimization"]
checks = {
    "datasets are the report's three":              list(_cfg.data.datasets) == SPEC["datasets"],
    "dataset_paths has exactly those three":        sorted(_cfg.data.dataset_paths) == sorted(SPEC["datasets"]),
    "K fixed at 50 (SAR)":                          _cfg.sar.k == SPEC["k"],
    "K fixed at 50 (baselines)":                    _cfg.baseline.k == SPEC["k"],
    "no K grid in the experiment plan":             "k_grid" not in _ps and int(_ps["fixed_k"]) == SPEC["k"],
    "MIGRES max iterations = 5":                    _cfg.migres.max_iter == SPEC["t_max"],
    "3 random + 7 guided trials":                   (_bo["init_points"], _bo["n_iter"]) == (SPEC["n_init"], SPEC["n_iter"]),
    "8 validation questions per dataset":           int(_ps["validation_subset_per_dataset"]) == SPEC["n_per_dataset"],
    "plan datasets match config datasets":          list(_ps["datasets"]) == SPEC["datasets"],
    "one seed (BO = holdout = run)":                _bo["seed"] == _cfg.data.validation_seed == _cfg.run.seed,
    "10% holdout of train":                         abs(_cfg.data.validation_fraction - 0.10) < 1e-12 and _cfg.data.corpus_source_split == "train",
    "LLM = gemini-3.5-flash-lite via OpenAI-compatible endpoint": _cfg.llm.model == SPEC["llm_model"] and _cfg.llm.provider == "openai",
    "LLM temperature = provider default (None)":    _cfg.llm.temperature is None,
    "embedding model = multi-qa-mpnet-base-cos-v1": _cfg.embedding.model_name == SPEC["embedding_model"],
    "objective = mean Recall@K":                    _bo["objective"] == "mean_final_recall_at_k",
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Config differs from the report - fix src/configs/ before running."

DATASETS = list(SPEC["datasets"])
K_FIXED, N_INIT, N_ITER = SPEC["k"], int(_bo["init_points"]), int(_bo["n_iter"])
N_TRIALS, SEED, N_PER_DS = N_INIT + N_ITER, int(_bo["seed"]), int(_ps["validation_subset_per_dataset"])
print(f"\nPlan: {len(DATASETS)} datasets x {N_PER_DS} questions, K={K_FIXED} (fixed), {N_TRIALS} trials "
      f"({N_INIT} random + {N_ITER} guided), seed={SEED}")

## 0. Setup — dependencies, models, datasets, corpus

Each step is **idempotent**: if the file or model is already present it is skipped, so re-running the notebook is cheap.

| Step | What it does | Output |
|---|---|---|
| 0.1 | Install missing Python packages | — |
| 0.2 | Load `.env`, locate the MIGRES repo, check credentials | env vars |
| 0.3 | Download the Sentence Transformer (SAR encoder) and the MIGRES reranker | Hugging Face cache |
| 0.4 | Download the **three** benchmark datasets and normalise them to the layout the loaders expect | `data/raw/*.json` |
| 0.5 | Validate every file with the project's own loaders | printed table |
| 0.6 | Build the pooled, deduplicated, chunked training corpus `D` (rebuilt automatically if it was built for another dataset set) | `data/processed/pooled_train_corpus.jsonl` |
| 0.6b | Select the 8 **labeled** validation questions per dataset | `TUNING_ITEMS` |
| 0.7 | Show where the tuned weights are written and who reads them | — |

In [ ]:
# 0.1 — Environment check only. Dependencies must come from the package extras; this notebook never mutates the environment.
import importlib.util
REQUIRED = ["torch", "transformers", "sentence_transformers", "bayes_opt", "nbformat", "requests", "datasets"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
print("Missing required setup packages:", missing or "none")
assert not missing, "Install the repository extras before running this notebook; the notebook does not install packages automatically."


In [ ]:
# 0.2  Environment: .env, MIGRES repo, credentials
import os
from sar_migres.utils.config import load_dotenv_file

os.chdir(ROOT)   # dataset/corpus paths in configs/default.yaml are relative to the project root
loaded = load_dotenv_file(ROOT / ".env", override=True)
print("Loaded from .env:", loaded)

# MIGRES repo: use the value from .env if it exists on this machine, otherwise look for the bundled copy.
candidates = [os.environ.get("MIGRES_REPO_DIR", ""), str(ROOT.parent / "MIGRES"), str(ROOT / "MIGRES")]
MIGRES_DIR = next((Path(c) for c in candidates if c and (Path(c) / "code" / "pipeline.py").exists()), None)
if MIGRES_DIR is not None:
    os.environ["MIGRES_REPO_DIR"] = str(MIGRES_DIR)
    print("MIGRES repo:", MIGRES_DIR)
else:
    print("MIGRES repo NOT FOUND. Tried:", [c for c in candidates if c])
    print("Clone it:  git clone https://github.com/<upstream>/MIGRES  and set MIGRES_REPO_DIR in .env")

# Names only - values are never printed.
for name in ("GEMINI_API_KEY", "OPENAI_BASE_URL", "LLM_MODEL", "MIGRES_REPO_DIR", "HF_TOKEN"):
    print(f"  {name:<16}", "set" if os.environ.get(name) else "NOT SET")
if not os.environ.get("GEMINI_API_KEY"):
    print("WARNING: GEMINI_API_KEY is not set - the MIGRES loop cannot call the LLM during tuning.")

In [ ]:
# 0.3  Download models (cached by huggingface_hub; re-runs are instant)
import time
from sar_migres.utils.config import load_config

_cfg0 = load_config(*BASE_CONFIG)
ST_MODEL = _cfg0.embedding.model_name            # SAR encoder (methodology: multi-qa-mpnet-base-cos-v1)
RERANKER = _cfg0.migres.ranker_model_path        # common MIGRES cross-encoder filter
EXPECTED_DIM = _cfg0.embedding.expected_dim

t0 = time.time()
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(ST_MODEL, device=DEVICE)
dim = st.get_sentence_embedding_dimension()
print(f"Sentence Transformer: {ST_MODEL}  dim={dim}  device={DEVICE}  ({time.time()-t0:.1f}s)")
assert EXPECTED_DIM in (None, dim), f"embedding dim {dim} != configured expected_dim {EXPECTED_DIM}"
_ = st.encode(["smoke test sentence"], normalize_embeddings=True)
del st

t0 = time.time()
from huggingface_hub import snapshot_download
rr_path = snapshot_download(RERANKER)
print(f"MIGRES reranker:     {RERANKER}  -> {rr_path}  ({time.time()-t0:.1f}s)")

# Tokenizer used to chunk the corpus (same model as the encoder)
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(ST_MODEL, use_fast=True)
print("Chunking tokenizer ready.")

# NLTK data used by the MIGRES filter / sentence splitting (harmless if already present)
try:
    import nltk
    for pkg in ("punkt", "punkt_tab"):
        nltk.download(pkg, quiet=True)
except Exception as e:
    print("NLTK data download skipped:", type(e).__name__, e)

In [ ]:
# 0.4a  Dataset download settings
import hashlib, gzip, io, shutil, zipfile
import requests

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

FORCE_REDOWNLOAD = False        # True = rebuild every data/raw/*.json even if it exists

# Size caps. The pooled corpus is capped at 100k passages (~33.3k per dataset), so the full
# training sets (HotpotQA alone is ~90k questions x 10 paragraphs) are not needed. Raise the caps
# if you want a larger pool of questions; None = keep everything.
MAX_TRAIN_PER_DATASET = None   # all train questions
MAX_DEV_PER_DATASET = None     # all dev questions
HOLDOUT_SEED = load_config(*BASE_CONFIG).data.validation_seed   # must equal the seed used by split_holdout_from_train

def _bucket(key: str) -> float:
    """Deterministic value in [0,1): the SAME hash the project uses to pick the 10% tuning holdout."""
    return int(hashlib.sha256(f"{HOLDOUT_SEED}:{key}".encode()).hexdigest()[:12], 16) / 16**12

def cap(rows, n, key="id"):
    """Deterministic, order-independent subsample of at most n rows.
    Rows are sorted by hash bucket, so the 10% tuning holdout (bucket < 0.10) sits at the FRONT of each
    file. The corpus builder consumes records in file order under a per-dataset quota, which guarantees
    that the passages of the tuning questions are inside the corpus D."""
    ordered = sorted(rows, key=lambda r: _bucket(str(r[key])))
    return ordered if n is None else ordered[:n]

def http_download(url, dest: Path, chunk=1 << 20):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not FORCE_REDOWNLOAD:
        print(f"  cached: {dest.name} ({dest.stat().st_size/1e6:.0f} MB)")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(tmp, "wb") as fh:
            for part in r.iter_content(chunk):
                fh.write(part); done += len(part)
                if total:
                    print(f"\r  {dest.name}: {done/1e6:.0f}/{total/1e6:.0f} MB", end="")
    print()
    tmp.replace(dest)
    return dest

def write_dataset(name, train, dev):
    """Write ONE JSON file per dataset: {"train": [...], "dev": [...]}.
    The project loaders pick the right list from the split key (datasets._read_records)."""
    out = RAW / FILES[name]
    out.write_text(json.dumps({"train": train, "dev": dev}, ensure_ascii=False), encoding="utf-8")
    print(f"  wrote {out.name}: train={len(train)} dev={len(dev)}  ({out.stat().st_size/1e6:.1f} MB)")

# File names must match configs/default.yaml -> data.dataset_paths
FILES = {k: Path(v).name for k, v in load_config(*BASE_CONFIG).data.dataset_paths.items()}
print(FILES)

In [ ]:
# 0.4b  HotpotQA (distractor setting) and MuSiQue
# HotpotQA is loaded from Hugging Face (hotpotqa/hotpot_qa, config "distractor": 90,447 train / 7,405 validation,
# the same content as the official CMU files). The CMU host (curtis.ml.cmu.edu, plain http) is only a fallback because
# it is often unreachable. Manual option: put hotpot_train_v1.1.json and hotpot_dev_distractor_v1.json into data/raw/_dl/.
def _need(name):
    return FORCE_REDOWNLOAD or not (RAW / FILES[name]).exists()

def _hotpot_from_hf(split):
    """Convert HF rows to the official layout the project loaders expect:
       _id, question, answer, context [[title,[sentences]]], supporting_facts [[title, sent_idx]]."""
    from datasets import load_dataset
    ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split=split)
    rows = []
    for r in ds:
        ctx, sf = r["context"], r["supporting_facts"]
        rows.append({"_id": r["id"], "question": r["question"], "answer": r["answer"],
                     "level": r.get("level"), "type": r.get("type"),
                     "context": [[t, list(s)] for t, s in zip(ctx["title"], ctx["sentences"])],
                     "supporting_facts": [[t, int(i)] for t, i in zip(sf["title"], sf["sent_id"])]})
    return rows

def _hotpot_from_cmu():
    base = "https://curtis.ml.cmu.edu/datasets/hotpot/"
    tr = http_download(base + "hotpot_train_v1.1.json", RAW / "_dl" / "hotpot_train_v1.1.json")
    dv = http_download(base + "hotpot_dev_distractor_v1.json", RAW / "_dl" / "hotpot_dev_distractor_v1.json")
    return json.loads(tr.read_text(encoding="utf-8")), json.loads(dv.read_text(encoding="utf-8"))

# ---- HotpotQA
if _need("hotpotqa"):
    print("HotpotQA ...")
    manual = [RAW / "_dl" / "hotpot_train_v1.1.json", RAW / "_dl" / "hotpot_dev_distractor_v1.json"]
    try:
        if all(p.exists() and p.stat().st_size > 0 for p in manual):      # files placed by hand win
            raise FileNotFoundError("using manually placed CMU files")
        train_rows, dev_rows = _hotpot_from_hf("train"), _hotpot_from_hf("validation")
        print("  source: Hugging Face hotpotqa/hotpot_qa (distractor)")
    except Exception as e:
        print(f"  Hugging Face route not used ({type(e).__name__}: {str(e)[:80]}) -> CMU files")
        train_rows, dev_rows = _hotpot_from_cmu()
    write_dataset("hotpotqa", cap(train_rows, MAX_TRAIN_PER_DATASET, "_id"), cap(dev_rows, MAX_DEV_PER_DATASET, "_id"))
    del train_rows, dev_rows
else:
    print("HotpotQA: present, skipping")

# ---- MuSiQue: fields id, question, answer, answer_aliases, paragraphs [{title, paragraph_text, is_supporting}]
if _need("musique"):
    print("MuSiQue ...")
    from datasets import load_dataset
    def _musique(split):
        ds = load_dataset("dgslibisey/MuSiQue", split=split)
        rows = []
        for r in ds:
            if r.get("answerable") is False:       # keep answerable questions only
                continue
            rows.append({"id": r["id"], "question": r["question"], "answer": r["answer"],
                         "answer_aliases": list(r.get("answer_aliases") or []),
                         "paragraphs": [{"idx": p.get("idx"), "title": p["title"], "paragraph_text": p["paragraph_text"],
                                         "is_supporting": bool(p["is_supporting"])} for p in r["paragraphs"]]})
        return rows
    write_dataset("musique", cap(_musique("train"), MAX_TRAIN_PER_DATASET), cap(_musique("validation"), MAX_DEV_PER_DATASET))
else:
    print("MuSiQue: present, skipping")

In [ ]:
# 0.4c  Natural Questions (DPR NQ training data; it carries the gold passages)
# Source order (the original DPR host dl.fbaipublicfiles.com is often unreachable):
#   1. files placed by hand in data/raw/_dl/ : nq-train.jsonl.gz + nq-dev.jsonl.gz (Tevatron)  or  biencoder-nq-train/dev.json.gz (DPR)
#   2. Hugging Face dataset Tevatron/wikipedia-nq  (raw .jsonl.gz files, ~1.4 GB train + 150 MB dev, cached by huggingface_hub)
#   3. the original DPR host
# Rows keep dataset=='nq' (see datasets._ODQA_TAG), short answers and the first 3 gold passages; negatives are dropped
# while streaming, so memory stays small.
# (WikiHop, TriviaQA and StrategyQA are NOT used: the report evaluates HotpotQA, MuSiQue and Natural Questions only.)
HF_NQ_REPO = "Tevatron/wikipedia-nq"
HF_NQ_FILES = {"train": "nq-train.jsonl.gz", "dev": "nq-dev.jsonl.gz"}
DPR_BASE = "https://dl.fbaipublicfiles.com/dpr/data/retriever/"
DPR_NQ_FILES = {"train": "biencoder-nq-train.json.gz", "dev": "biencoder-nq-dev.json.gz"}

def _nq_row(split, i, question, answers, passages):
    return {"id": f"nq-{split}-{i}", "dataset": "nq", "question": str(question).strip(),
            "answer": [str(a) for a in answers],
            "positive_ctxs": [{"title": p.get("title", "") or "", "text": p["text"]} for p in passages[:3]]}

def _nq_from_tevatron(path, split, limit):
    """Tevatron format, one JSON object per line: query, answers, positive_passages[{docid,title,text}], ..."""
    out = []
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if not line.strip():
                continue
            rec = json.loads(line)
            pos, ans = rec.get("positive_passages") or [], rec.get("answers") or []
            if not str(rec.get("query", "")).strip() or not ans or not pos:
                continue
            out.append(_nq_row(split, i, rec["query"], ans, pos))
            if limit and len(out) >= limit:
                break
    return out

def _nq_from_dpr(path, split, limit):
    """Original DPR format: one big JSON list with question, answers, positive_ctxs[{title,text}], ..."""
    import ijson
    out = []
    with gzip.open(path, "rb") as fh:
        for i, rec in enumerate(ijson.items(fh, "item", use_float=True)):
            pos, ans = rec.get("positive_ctxs") or [], rec.get("answers") or []
            if not str(rec.get("question", "")).strip() or not ans or not pos:
                continue
            out.append(_nq_row(split, i, rec["question"], ans, pos))
            if limit and len(out) >= limit:
                break
    return out

def _nq_rows(split, limit):
    dl = RAW / "_dl"
    for fn, reader in ((HF_NQ_FILES[split], _nq_from_tevatron), (DPR_NQ_FILES[split], _nq_from_dpr)):
        if (dl / fn).exists() and (dl / fn).stat().st_size > 0:                 # 1. placed by hand / cached earlier
            print(f"  {split}: using {fn} from data/raw/_dl/")
            return reader(dl / fn, split, limit)
    try:                                                                      # 2. Hugging Face
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=HF_NQ_REPO, filename=HF_NQ_FILES[split], repo_type="dataset")
        print(f"  {split}: Hugging Face {HF_NQ_REPO}/{HF_NQ_FILES[split]}")
        return _nq_from_tevatron(p, split, limit)
    except Exception as e:
        print(f"  {split}: Hugging Face route failed ({type(e).__name__}: {str(e)[:80]}) -> DPR host")
    p = http_download(DPR_BASE + DPR_NQ_FILES[split], dl / DPR_NQ_FILES[split])   # 3. original DPR host
    return _nq_from_dpr(p, split, limit)

if _need("natural_questions"):
    print("natural_questions ...")
    nq_train = _nq_rows("train", MAX_TRAIN_PER_DATASET)
    nq_dev = _nq_rows("dev", MAX_DEV_PER_DATASET)
    write_dataset("natural_questions", cap(nq_train, None), cap(nq_dev, None))
    del nq_train, nq_dev
else:
    print("natural_questions: present, skipping")

In [ ]:
# 0.5  Validate every dataset through the project's own loaders
from sar_migres.datasets.loader import load_phase_questions

cfg = load_config(*BASE_CONFIG)
assert list(cfg.data.datasets) == DATASETS
rows = []
for ds in DATASETS:
    path = ROOT / cfg.data.dataset_paths[ds]
    if not path.exists():
        rows.append((ds, "MISSING", "-", "-")); continue
    tune = load_phase_questions(ds, path, "tuning", validation_fraction=cfg.data.validation_fraction, validation_seed=cfg.data.validation_seed)
    final = load_phase_questions(ds, path, "final", final_split=cfg.data.final_split)
    ids_t, ids_f = {x["id"] for x in tune}, {x["id"] for x in final}
    assert not (ids_t & ids_f), f"{ds}: tuning and final questions overlap!"
    rows.append((ds, "ok", len(tune), len(final)))

print(f"{'dataset':<20}{'status':<9}{'tuning (10% train)':<20}{'final (dev)'}")
for r in rows:
    print(f"{r[0]:<20}{r[1]:<9}{str(r[2]):<20}{r[3]}")
if any(r[1] != "ok" for r in rows):
    raise RuntimeError("Some dataset files are missing - re-run the download cells above.")

In [ ]:
# 0.6  Build the pooled training corpus D (train-split passages only; deduplicated; <=350 tokens, 50 overlap; <=100k)
# The corpus is rebuilt automatically if the existing one was built for a different dataset set
# (e.g. the old six-dataset corpus), because D must contain passages of the three report datasets only.
from sar_migres.corpus.builder import build_pooled_corpus

CORPUS_PATH = ROOT / cfg.data.corpus_path
MANIFEST_PATH = CORPUS_PATH.with_suffix(CORPUS_PATH.suffix + ".manifest.json")
REBUILD_CORPUS = False          # True = force a rebuild

stale = False
if CORPUS_PATH.exists() and MANIFEST_PATH.exists():
    _old = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    stale = (set(_old.get("per_dataset", {})) != set(DATASETS)
             or _old.get("max_documents") != cfg.data.max_documents
             or _old.get("source_split") != cfg.data.corpus_source_split)
    if stale:
        print("Existing corpus was built for", sorted(_old.get("per_dataset", {})), "-> rebuilding for", sorted(DATASETS))
elif CORPUS_PATH.exists():
    stale = True
    print("Existing corpus has no manifest -> rebuilding")

if CORPUS_PATH.exists() and not REBUILD_CORPUS and not stale:
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print("Corpus already built for the three report datasets - skipping.")
else:
    manifest = build_pooled_corpus(
        {ds: ROOT / cfg.data.dataset_paths[ds] for ds in DATASETS},
        output=CORPUS_PATH, source_split=cfg.data.corpus_source_split, max_documents=cfg.data.max_documents,
        max_tokens=cfg.data.corpus_chunk_max_tokens, overlap_tokens=cfg.data.corpus_chunk_overlap_tokens,
        tokenizer_name=ST_MODEL)
assert set(manifest["per_dataset"]) == set(DATASETS), manifest["per_dataset"]
print("passages:", manifest["n_documents"], "| tokenizer:", manifest["tokenizer"])
print("per dataset:", manifest["per_dataset"])

In [ ]:
# 0.6b  The 8 labeled validation questions per dataset (report Sec. 5.6.1)
# Recall@K needs gold evidence mapped to corpus ids. Questions without usable labels are excluded (Sec. 5.6.4).
#  - HotpotQA : gold titles from supporting_facts           -> corpus passages with that title
#  - MuSiQue  : paragraphs flagged is_supporting             -> same title + same text
#  - Natural Questions: positive_ctxs (DPR gold passages)    -> same title + same text
# Title+text matching is needed because the corpus builder re-chunks and lower-cases passages, so ids and
# raw strings no longer match the dataset's own ids. Candidates are the first questions of the deterministic
# 10% holdout (hash order); the first 8 that have labels in the corpus are used, the same 8 in every trial.
import re
from collections import defaultdict
from sar_migres.corpus.models import Corpus

CANDIDATES_PER_DATASET = 60
OVERLAP = 0.80                         # share of a corpus chunk's words that must occur in the gold passage
_tok = re.compile(r"\w+", re.UNICODE)
_title = lambda t: " ".join(_tok.findall(str(t).lower()))
_words = lambda t: set(_tok.findall(str(t).lower()))

def gold_passages(item):
    ds = item["dataset"]
    if ds == "hotpotqa":               # title-addressable, as in the project loader
        return [(str(x[0]), None) for x in (item.get("supporting_facts") or []) if isinstance(x, (list, tuple)) and x]
    if ds == "musique":
        return [(str(p.get("title", "")), str(p.get("paragraph_text", ""))) for p in (item.get("paragraphs") or [])
                if isinstance(p, dict) and p.get("is_supporting")]
    if ds == "natural_questions":
        return [(str(c.get("title", "")), str(c.get("text", ""))) for c in (item.get("positive_ctxs") or [])]
    return []

cands = {ds: load_phase_questions(ds, ROOT / cfg.data.dataset_paths[ds], "tuning",
                                  validation_fraction=cfg.data.validation_fraction,
                                  validation_seed=cfg.data.validation_seed)[:CANDIDATES_PER_DATASET] for ds in DATASETS}
need = {_title(t) for items in cands.values() for it in items for t, _ in gold_passages(it)}

corpus_lbl = Corpus.from_file(CORPUS_PATH, embed_template=cfg.embedding.embed_template, lazy=True,
                              max_malformed_fraction=cfg.data.max_malformed_fraction)
by_title = defaultdict(list)           # one scan of D; only passages whose title can be gold are kept
for d in corpus_lbl.iter_documents():
    t = _title(d.title)
    if t in need:
        by_title[t].append((d.doc_id, _words(d.text)))

def resolve(item):
    ids = set()
    for title, text in gold_passages(item):
        gold_words = _words(text) if text is not None else None
        for did, words in by_title.get(_title(title), []):
            if gold_words is None or (words and len(words & gold_words) / len(words) >= OVERLAP):
                ids.add(did)
    return ids

TUNING_ITEMS, coverage = {}, []
for ds in DATASETS:
    chosen, skipped = [], 0
    for it in cands[ds]:
        ids = resolve(it)
        if not ids:
            skipped += 1; continue
        it = dict(it); it["supporting_doc_ids"] = sorted(ids)      # explicit labels are honoured by relevant_doc_ids
        chosen.append(it)
        if len(chosen) == N_PER_DS:
            break
    if len(chosen) < N_PER_DS:
        raise RuntimeError(f"{ds}: only {len(chosen)} of {N_PER_DS} tuning questions have gold passages in the corpus "
                           f"(skipped {skipped}). Rebuild the corpus / check the dataset file.")
    TUNING_ITEMS[ds] = chosen
    coverage.append((ds, len(cands[ds][:len(chosen) + skipped]), skipped, len(chosen),
                     sum(len(x["supporting_doc_ids"]) for x in chosen) / len(chosen)))

print(f"{'dataset':<20}{'examined':<10}{'no labels':<11}{'used':<6}{'gold passages / question'}")
for r in coverage:
    print(f"{r[0]:<20}{r[1]:<10}{r[2]:<11}{r[3]:<6}{r[4]:.1f}")
print("\nTuning questions:", {k: [x['id'] for x in v] for k, v in TUNING_ITEMS.items()})

In [ ]:
# True = run the full Bayesian tuning: 10 trials x 3 datasets x 8 questions, real LLM calls.
# Set to False for a preflight that only prints the plan.
RUN_EXPERIMENT = False
SELECTED_PATH = ROOT / "outputs/experiments/01_parameter_selection/selected_config.json"
RUNS_ROOT = ROOT / "outputs/experiments/01_parameter_selection/bayes_runs"
TRIAL_LOG_PATH = ROOT / "outputs/experiments/01_parameter_selection/trial_log.jsonl"

In [ ]:
# 0.7  Where the tuned weights go, and who reads them
# Written by the tuning cell below -> read by notebooks 02, 03, 06 and 07 through
# experiment_runner.frozen_setting(path)  ->  (selected_k, SARWeights(*selected_weights)).
# selected_k is always the fixed K = 50 (it is not tuned); it is stored so those notebooks keep working.
print("Tuned weights are saved to:\n  ", SELECTED_PATH)
print("Exists now:", SELECTED_PATH.exists())
for nb in ("02_baseline_comparison", "03_ablation_study", "06_downstream_rag_robustness", "07_computational_cost"):
    f = next((p for p in (ROOT / "notebooks" / f"{nb}.ipynb", ROOT / f"{nb}.ipynb") if p.exists()), None)
    if f is None:
        print(f"  {nb:<32} notebook file not found"); continue
    print(f"  {nb:<32} reads selected_config.json:", "01_parameter_selection/selected_config.json" in f.read_text(encoding="utf-8"))

## Tuning

Only α, β, γ are searched, through the two free values `(u, v) ∈ [0,1]²` with `α = u, β = (1−u)v, γ = (1−u)(1−v)`, so every candidate satisfies Eq. 8. **K stays 50 in every trial.**

* Trials 1–3 use random weights; trials 4–10 are proposed by the Gaussian-process optimizer (the same `seed` fixes the random start points).
* Each trial runs the full MIGRES loop with SAR on the same 24 labeled validation questions (8 per dataset).
* Trial score = mean over datasets of the final-iteration raw-ranking Recall@50.
* The best trial's weights are written to `selected_config.json` and frozen.

In [ ]:
# 0.1b — Environment check only (dependencies come from the package extras; notebooks never mutate the environment).
import importlib.util
print("accelerate:", "installed" if importlib.util.find_spec("accelerate") else "MISSING — install sar-migres[notebook,migres]")


In [ ]:
import os
import nltk

nltk_dir = r"str(ROOT / 'data' / 'cache' / 'nltk_data')"
os.makedirs(nltk_dir, exist_ok=True)

if nltk_dir not in nltk.data.path:
    nltk.data.path.insert(0, nltk_dir)

nltk.download("punkt", download_dir=nltk_dir, quiet=False)
nltk.download("punkt_tab", download_dir=nltk_dir, quiet=False)

print(nltk.data.find("tokenizers/punkt"))

In [ ]:
import math, time, threading, logging, sys
from datetime import datetime
from sar_migres.experiments.iterative import run_pipeline_experiment, retrieval_metrics_from_run
from sar_migres.experiments.parameter_selection import optimize_weights
from sar_migres.scoring.weights import SARWeights

# ---------------------------------------------------------------- settings
RESUME = True            # True = reuse finished trials from trial_log.jsonl (same seed -> same weights)
HEARTBEAT_SECONDS = 20   # how often a "still running" line is printed

logging.getLogger("sar_migres").setLevel(logging.INFO)   # show the package's own progress messages

def log(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {sec % 3600 // 60:02d}m {sec % 60:02d}s" if sec >= 3600 else f"{sec // 60}m {sec % 60:02d}s"

def _count_lines(path):
    try:
        with open(path, "rb") as fh:
            return sum(1 for _ in fh)
    except OSError:
        return 0

def _newest_run_dir(name):
    found = [p for p in RUNS_ROOT.glob(name + "*") if p.is_dir()]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None

class Heartbeat(threading.Thread):
    """Prints a status line every few seconds by reading the run folder. It never touches the pipeline."""
    def __init__(self, label, run_name, total):
        super().__init__(daemon=True)
        self.label, self.run_name, self.total = label, run_name, total
        self.stop_event, self.t0 = threading.Event(), time.time()

    def run(self):
        while not self.stop_event.wait(HEARTBEAT_SECONDS):
            d = _newest_run_dir(self.run_name)
            if d is None:
                state = "starting (loading models / syncing vector store)"
            else:
                q = _count_lines(d / "results.jsonl")
                it = _count_lines(d / "traces.jsonl")
                state = f"questions finished {q}/{self.total} | iterations traced {it}"
            gpu = ""
            try:
                import torch
                if torch.cuda.is_available():
                    gpu = f" | GPU mem {torch.cuda.memory_allocated() / 1e9:.1f} GB"
            except Exception:
                pass
            log(f"    ... {self.label} running {fmt(time.time() - self.t0)} | {state}{gpu}")

    def stop(self):
        self.stop_event.set()

# ---------------------------------------------------------------- plan
print("Scope:", PLAN["parameter_selection"]["bayesian_optimization"]["scope"])
print("Budget:", PLAN["parameter_selection"]["bayesian_optimization"]["budget_note"])
print("Questions per dataset:", {k: len(v) for k, v in TUNING_ITEMS.items()}, flush=True)

def _overrides(w, ds, name):
    return {"retrieval": {"method": "sar"},
            "sar": {"alpha": float(w.alpha), "beta": float(w.beta), "gamma": float(w.gamma), "k": K_FIXED,
                    "weighting": "static", "dynamic_rule": ""},
            "data": {"dataset": ds, "dataset_path": cfg.data.dataset_paths[ds], "split": "train"},
            "run": {"name": name, "output_dir": str(RUNS_ROOT), "resume": False}}

if RUN_EXPERIMENT:
    # fail early (before any LLM call) if the overrides would not give K=50 / the Gemini overlay
    _probe = load_config(*BASE_CONFIG, overrides=_overrides(SARWeights(0.4, 0.3, 0.3), DATASETS[0], "probe"))
    assert _probe.sar.k == K_FIXED and _probe.llm.model == SPEC["llm_model"] and _probe.llm.temperature is None

    # ---- finished trials from an earlier run (optional resume)
    cached = {}
    if RESUME and TRIAL_LOG_PATH.exists():
        for line in TRIAL_LOG_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                row = json.loads(line)
                cached[int(row["trial"])] = row
    if cached:
        log(f"Resume: found {len(cached)} finished trial(s) in the log; they are reused if the weights match.")

    trial_log, counter = [], {"n": 0}
    TRIAL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    TRIAL_LOG_PATH.write_text("", encoding="utf-8")
    run_start, trial_seconds = time.time(), []

    def _save_row(row):
        trial_log.append(row)
        with TRIAL_LOG_PATH.open("a", encoding="utf-8") as fh:                    # survives a crash
            fh.write(json.dumps(row) + "\n")

    def objective(w):
        counter["n"] += 1
        n, t0, per_ds = counter["n"], time.time(), {}
        phase = "random_init" if n <= N_INIT else "bayes_guided"

        old = cached.get(n)
        if old and all(abs(a - b) < 1e-9 for a, b in zip(old["weights"], w.as_tuple())):
            _save_row(old)
            log(f"trial {n:>2}/{N_TRIALS} [{phase}] reused from log  mean={old['objective']:.4f}")
            return float(old["objective"])

        log(f"=== trial {n}/{N_TRIALS} [{phase}] a={w.alpha:.3f} b={w.beta:.3f} g={w.gamma:.3f} ===")
        for k, (ds, items) in enumerate(TUNING_ITEMS.items(), 1):
            name = f"trial{n:02d}_{ds}"
            log(f"  trial {n} | dataset {k}/{len(TUNING_ITEMS)}: {ds} ({len(items)} questions) started")
            hb = Heartbeat(f"trial {n} {ds}", name, len(items))
            hb.start()
            t_ds = time.time()
            try:
                pipe, _ = run_pipeline_experiment(BASE_CONFIG, items, overrides=_overrides(w, ds, name),
                                                  experiment_name=name)
            finally:
                hb.stop()
            m = retrieval_metrics_from_run(pipe.run_dir, items, pipe.corpus)
            per_ds[ds] = {"recall_at_k": float(m["final_mean"]["recall_at_k"]),
                          "n_questions_scored": int(m["n_questions_with_relevance_annotations"]),
                          "n_questions_expected": len(items), "run_dir": str(pipe.run_dir)}
            if per_ds[ds]["n_questions_scored"] == 0 or math.isnan(per_ds[ds]["recall_at_k"]):
                raise RuntimeError(f"trial {n}, {ds}: no question could be scored - the objective would silently ignore this dataset.")
            if per_ds[ds]["n_questions_scored"] < len(items):
                log(f"  WARNING trial {n} {ds}: only {per_ds[ds]['n_questions_scored']}/{len(items)} questions scored (failed LLM calls?)")
            log(f"  trial {n} | {ds} done in {fmt(time.time() - t_ds)} | Recall@{K_FIXED}={per_ds[ds]['recall_at_k']:.3f} "
                f"({per_ds[ds]['n_questions_scored']}/{len(items)} questions scored)")

        score = sum(d["recall_at_k"] for d in per_ds.values()) / len(per_ds)       # mean Recall@K over the 3 datasets
        secs = time.time() - t0
        _save_row({"trial": n, "phase": phase, "weights": [float(x) for x in w.as_tuple()],
                   "objective": float(score), "per_dataset": per_ds, "seconds": round(secs, 1)})
        trial_seconds.append(secs)
        left = N_TRIALS - n
        eta = left * (sum(trial_seconds) / len(trial_seconds))
        log(f"trial {n:>2}/{N_TRIALS} [{phase:<11}] a={w.alpha:.3f} b={w.beta:.3f} g={w.gamma:.3f} "
            + " ".join(f"{d[:4]}={v['recall_at_k']:.3f}" for d, v in per_ds.items())
            + f"  mean={score:.4f}  | took {fmt(secs)} | about {fmt(eta)} left ({left} trials)")
        return score

    log(f"Starting Bayesian optimization: {N_TRIALS} trials ({N_INIT} random + {N_ITER} guided).")
    best, best_score, history = optimize_weights(objective, init_points=N_INIT, n_iter=N_ITER, seed=SEED)
    log(f"Search finished in {fmt(time.time() - run_start)}.")

    # ---- consistency checks on the search itself
    assert len(trial_log) == N_TRIALS == len(history), (len(trial_log), len(history))
    assert [t["phase"] for t in trial_log].count("random_init") == N_INIT
    assert [t["phase"] for t in trial_log].count("bayes_guided") == N_ITER
    top = max(trial_log, key=lambda r: r["objective"])
    sel = [float(x) for x in best.as_tuple()]
    assert all(abs(a - b) < 1e-9 for a, b in zip(sel, top["weights"])), "optimizer best != best trial"
    assert min(sel) >= 0 and abs(sum(sel) - 1.0) < 1e-9, sel                          # Eq. 8

    result = {
        "objective": "mean_final_recall_at_k",
        "scope": "global_across_three_datasets",
        "split": "10% train holdout (labeled questions only)",
        "selected_k": K_FIXED,
        "k_note": "fixed (original MIGRES candidate pool); not tuned",
        "selected_weights": sel,                       # [alpha, beta, gamma]
        "best_objective": float(best_score),
        "best_trial": top["trial"],
        "protocol": {"datasets": DATASETS, "questions_per_dataset": N_PER_DS,
                     "question_ids": {k: [x["id"] for x in v] for k, v in TUNING_ITEMS.items()},
                     "n_trials": N_TRIALS, "n_random": N_INIT, "n_guided": N_ITER, "seed": SEED,
                     "holdout_fraction": cfg.data.validation_fraction, "holdout_seed": cfg.data.validation_seed,
                     "max_iterations": cfg.migres.max_iter, "weighting": "static",
                     "llm_model": _probe.llm.model, "llm_temperature": "provider default",
                     "embedding_model": cfg.embedding.model_name, "corpus_documents": manifest["n_documents"],
                     "corpus_per_dataset": manifest["per_dataset"]},
        "trials": trial_log,
    }
    SELECTED_PATH.parent.mkdir(parents=True, exist_ok=True)
    SELECTED_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("\nSelected (fixed for all later experiments):")
    print(json.dumps({k: result[k] for k in ("objective", "scope", "selected_k", "selected_weights", "best_objective", "best_trial")}, indent=2))
    print("Saved:", SELECTED_PATH)
    print("Note: seeds fix the holdout and the random start points; Gemini runs at its default temperature, "
          "so a re-run reproduces the protocol but not necessarily identical recall values.")
else:
    print(f"Preflight only: {len(DATASETS)} datasets x {N_PER_DS} questions, K={K_FIXED}, {N_TRIALS} trials "
          f"({N_INIT} random + {N_ITER} guided), seed={SEED}. Expected LLM-driven MIGRES runs: {N_TRIALS * len(DATASETS)} batches of {N_PER_DS} questions.")

In [ ]:
# Plot the fixed 10 Bayesian-optimization trial objectives after a completed tuning run.
from sar_migres.evaluation.plotting import save_line_plot
if SELECTED_PATH.exists():
    tuned=json.loads(SELECTED_PATH.read_text(encoding='utf-8'))
    trial_values=[float(t['objective']) for t in tuned.get('trials',[]) if t.get('objective') is not None]
    if trial_values:
        save_line_plot({'validation mean Recall@K':trial_values},'Mean Recall@K','SAR weight selection trials',
                       SELECTED_PATH.parent/'plots'/'tuning_objective.png',xlabel='Trial')
        print('Tuning plot:',SELECTED_PATH.parent/'plots'/'tuning_objective.png')
else:
    print('Tuning plot will be saved after the selected_config.json artifact exists.')
